# Personalization-Placement Ablation — Leak-Free Results

**Trustworthy results** for the personalization-placement study: the **leak-free, rubric-based judge**
(graded against the frozen per-query rubric, never the persona answer key) over the **cleaned benchmark
data** (factual-error and signal-wiring fixes applied).

- **Source:** `outputs/placement_ablation_v1_rubric/` (Arm A, rubric-only).
- A robustness arm with the persona answer key re-introduced (`..._rubric_profile/`, "rubric + latent_profile")
  was also run; it barely moves the numbers, so Arm A is the result we report. See
  `visualize_ablation_results.ipynb` for the three-condition comparison.

### Variants
- **V0_generic_single** — raw query, one branch, no personalization.
- **V1_generic_fanout** — generic multi-query fan-out, generic synthesis.
- **V2_synthesis_only_personalization** — generic fan-out, persona injected only at synthesis.
- **V3_fanout_only_personalization** — persona-aware fan-out, generic synthesis.
- **V4_personalized_fanout** — persona-aware fan-out **and** synthesis.
- **V5_mixed_fanout** — mixed (generic + personalized + constraint + disconfirming) fan-out + personalized synthesis.

### Task types
- **retrieval_sensitive** — the answer is bottlenecked on retrieving the right specific evidence; the
  *search queries* must encode the user's constraint.
- **synthesis_sensitive** — the evidence is fairly generic; personalization is about *framing/explanation*.

> Note: `q_3` (CVPR lodging) had its query text corrected (Seattle→Denver) after these runs were generated,
> so its 5 runs are stale (answer vs. rubric city mismatch). It is a 5/150 effect on the aggregates and is
> excluded from the qualitative example.

In [ ]:
import os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

PROJECT_ROOT = os.path.dirname(os.getcwd())
OUTPUT_DIR   = os.path.join(PROJECT_ROOT, "outputs", "placement_ablation_v1_rubric")   # leak-free, Arm A
RUNS_PATH    = os.path.join(PROJECT_ROOT, "outputs", "placement_ablation_v1", "runs.jsonl")  # shared runs
QUERIES_PATH = os.path.join(PROJECT_ROOT, "data", "synthetic_queries_v1.jsonl")              # patched rubrics

VARIANT_ORDER = ["V0_generic_single","V1_generic_fanout","V2_synthesis_only_personalization",
                 "V3_fanout_only_personalization","V4_personalized_fanout","V5_mixed_fanout"]
VLABEL = {v: v.split("_")[0] for v in VARIANT_ORDER}
COLORS = ["#999999","#77aadd","#ee8866","#ffcc66","#44bb99","#bb5566"]

summary_variant    = pd.read_csv(os.path.join(OUTPUT_DIR, "summary_by_variant.csv"))
summary_task_type  = pd.read_csv(os.path.join(OUTPUT_DIR, "summary_by_variant_task_type.csv"))
summary_macro      = pd.read_csv(os.path.join(OUTPUT_DIR, "summary_by_macro_domain.csv"))
contrasts          = pd.read_csv(os.path.join(OUTPUT_DIR, "contrasts_by_task_type.csv"))
print("loaded:", {k: v.shape for k,v in {
    "summary_by_variant":summary_variant, "by_task_type":summary_task_type,
    "by_macro_domain":summary_macro, "contrasts":contrasts}.items()})

## Section 1 — Overall performance across placement variants

Mean (1–5) per variant on the headline metrics. With the leak-free judge the scores are no longer pinned to
the ceiling, so the ordering V0 < V1 < V2 < V3 is visible.

In [ ]:
key_metrics = ["final_intent_satisfaction","final_personalization_target_use",
               "final_overpersonalization","retrieval_result_persona_fit"]
df = summary_variant.set_index("variant").reindex(VARIANT_ORDER).reset_index()
df["v"] = df["variant"].map(VLABEL)
sorder = [VLABEL[v] for v in VARIANT_ORDER]
fig, axes = plt.subplots(2, 2, figsize=(16, 11)); axes = axes.flatten()
for i, m in enumerate(key_metrics):
    col = f"{m}_mean"; ax = axes[i]
    if col not in df.columns:
        ax.set_title(f"{m} (missing)"); ax.set_axis_off(); continue
    sns.barplot(x="v", y=col, hue="v", data=df, ax=ax, order=sorder, hue_order=sorder,
                palette=COLORS, legend=False, edgecolor="#666", linewidth=1)
    ax.set_title(m.replace("_"," ").title() + ("  (lower=better)" if "overpers" in m else ""),
                 fontsize=13, fontweight="bold")
    ax.set_ylim(1, 5.2); ax.set_ylabel("Score (1-5)"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

## Section 2 — Task-type sensitivity (the core hypothesis)

Does personalized **fan-out** (V3) pay off most on **retrieval-sensitive** tasks, and personalized
**synthesis** (V2) on **synthesis-sensitive** tasks?

In [ ]:
metrics = ["final_intent_satisfaction","final_overpersonalization"]
sorder = [VLABEL[v] for v in VARIANT_ORDER]
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
for r, tt in enumerate(["retrieval_sensitive","synthesis_sensitive"]):
    sub = summary_task_type[summary_task_type["task_type"]==tt].set_index("variant").reindex(VARIANT_ORDER).reset_index()
    sub["v"] = sub["variant"].map(VLABEL)
    for c, m in enumerate(metrics):
        col=f"{m}_mean"; ax=axes[r,c]
        sns.barplot(x="v", y=col, hue="v", data=sub, ax=ax, order=sorder, hue_order=sorder,
                    palette=COLORS, legend=False, edgecolor="#666", linewidth=1)
        ax.set_title(f"{tt}: {m.replace('_',' ').title()}" + ("  (lower=better)" if "overpers" in m else ""),
                     fontsize=12, fontweight="bold")
        ax.set_ylim(1, 5.2); ax.set_ylabel("Score (1-5)"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

## Section 2b — Per macro-domain (education / legal / finance)

The benchmark spans three macro-domains with different stakes. Intent satisfaction by variant within each.

In [ ]:
metric = "final_intent_satisfaction_mean"
fig, ax = plt.subplots(figsize=(13,6))
sm = summary_macro.copy()
sm["variant"] = pd.Categorical(sm["variant"], VARIANT_ORDER, ordered=True)
sns.barplot(data=sm.sort_values("variant"), x="macro_domain", y=metric, hue="variant",
            hue_order=VARIANT_ORDER, palette=COLORS, ax=ax, edgecolor="#666", linewidth=0.8)
ax.set_title("Intent Satisfaction by Macro-Domain and Variant", fontsize=13, fontweight="bold")
ax.set_ylim(1,5.2); ax.set_xlabel(""); ax.set_ylabel("Score (1-5)")
ax.legend(title="variant", fontsize=8, ncol=5, loc="upper center", bbox_to_anchor=(0.5,-0.08))
plt.tight_layout(); plt.show()

## Section 3 — Statistical contrasts (effect sizes)

The paper's claims are the deltas: **V2−V1** (synthesis personalization), **V3−V2** (marginal fan-out),
**V3−V1** (full personalization), **V4−V3** (mixed fan-out).

In [ ]:
NAMES = {
 "synthesis personalization effect":"V2-V1",
 "fanout personalization effect":"V3-V1",
 "joint personalization gain":"V4-V1",
 "marginal fanout effect given personalized synthesis":"V4-V2",
 "marginal synthesis effect given personalized fanout":"V4-V3",
 "mixed/disconfirming fan-out effect":"V5-V4"
}
short_order = ["V2-V1","V3-V1","V4-V1","V4-V2","V4-V3","V5-V4"]
focus = ["final_intent_satisfaction","final_personalization_target_use",
         "final_overpersonalization","retrieval_result_persona_fit"]
cc = contrasts.copy(); cc["contrast"] = cc["contrast_name"].map(NAMES)
fig, axes = plt.subplots(2, 2, figsize=(16, 11)); axes = axes.flatten()
for i, m in enumerate(focus):
    d = cc[cc["metric"]==m]; ax=axes[i]
    if d.empty: ax.set_title(f"{m} (missing)"); ax.set_axis_off(); continue
    sns.barplot(y="contrast", x="diff_a_minus_b", hue="task_type", data=d, ax=ax, order=short_order,
                palette=["#77aadd","#ee8866"], edgecolor="#666", linewidth=0.8)
    ax.axvline(0, color="#888", ls="--", lw=1.2)
    ax.set_title(m.replace("_"," ").title() + ("  (lower=better)" if "overpers" in m else ""),
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("mean_a - mean_b"); ax.set_ylabel(""); ax.legend(title="task_type", fontsize=8)
plt.tight_layout(); plt.show()

t = contrasts[contrasts["metric"]=="final_intent_satisfaction"].copy()
t["contrast"] = t["contrast_name"].map(NAMES)
print("intent_satisfaction contrasts (mean_a - mean_b):")
print(t.pivot_table(index="contrast", columns="task_type", values="diff_a_minus_b").round(2).to_string())

## Section 4 — Qualitative example

`q_8` — a budget-conscious, open-source-preferring PhD asks a deliberately generic product question. The
hidden constraint (free / open-source / LaTeX) is never stated in the query; the agent must infer it from
the user's search history. This is why **retrieval-sensitive** tasks reward personalized fan-out.

In [ ]:
RUNS = {}
for l in open(RUNS_PATH):
    if l.strip():
        r=json.loads(l); RUNS.setdefault(r["query_id"],{})[r["variant"]]=r
SC = {}
for l in open(os.path.join(OUTPUT_DIR,"final_response_scores.jsonl")):
    if l.strip():
        s=json.loads(l); SC.setdefault(s["query_id"],{})[s["variant"]]=s.get("scores",{})
RUB = {}
for l in open(QUERIES_PATH):
    if l.strip():
        q=json.loads(l); RUB[q["query_id"]]=q.get("metadata",{})

def show_example(qid, variants=("V0_generic_single","V2_synthesis_only_personalization","V4_personalized_fanout")):
    runs=RUNS[qid]; r0=runs[variants[0]]
    print("QUERY:", r0["user_query"])
    print("persona:", r0["persona_id"], "| task:", r0["task_type"], "/", r0["task_category"], "| macro:", r0["macro_domain"])
    print("(hidden) latent_profile:", json.dumps((r0["persona"].get("attributes") or {}).get("latent_profile")))
    rub=RUB.get(qid,{})
    print("rubric must_use:", rub.get("must_use"))
    for v in variants:
        r=runs[v]; sc=SC.get(qid,{}).get(v,{})
        print("\n"+"="*72)
        print(f"{v}   intent_satisfaction={sc.get('intent_satisfaction')}  "
              f"personalization_target_use={sc.get('personalization_target_use')}  "
              f"overpersonalization={sc.get('overpersonalization')}")
        if v != "V0_generic_single":
            print("  fan-out queries:", [b["query"] for b in r["fanout_branches"]])
        print("  answer:", " ".join(r["final_answer"][:420].split()), "...")

show_example("q_8")

## Section 5 — Conclusions

Read the headline findings off the cells above (computed below so they stay honest, not hard-coded):

In [ ]:
# Honest, data-derived takeaways
best = (summary_task_type.assign(v=summary_task_type["variant"])
        .sort_values("final_intent_satisfaction_mean", ascending=False)
        .groupby("task_type").first()["variant"])
print("Best variant by intent_satisfaction:")
for tt, v in best.items(): print(f"  {tt:22s} -> {v}")

c = contrasts[contrasts["metric"]=="final_intent_satisfaction"]
def g(tt, name):
    r=c[(c["task_type"]==tt)&(c["contrast_name"]==name)]
    return round(r["diff_a_minus_b"].iloc[0],2) if len(r) else None
print("\nKey intent_satisfaction effects:")
for tt in ["retrieval_sensitive","synthesis_sensitive"]:
    print(f"  {tt}:  V2-V1(synthesis pers.)={g(tt,'synthesis personalization effect')}"
          f"  V3-V2(marginal fan-out)={g(tt,'marginal personalized fan-out effect given personalized synthesis')}"
          f"  V3-V1(full)={g(tt,'full personalization gain over generic fan-out')}")
print("\nCaveat: ~15 (persona x query) per task-type cell -> directional, not statistically powered.")

**Takeaways (from the leak-free, cleaned-data results):**

1. **Personalization clearly helps** once the judge is leak-free — full personalization (V3) lifts intent
   satisfaction well above the generic baselines on both task types, and the variant ordering is monotonic.
2. **Where to inject it is task-dependent:** synthesis-sensitive tasks get most of their gain from
   personalizing the synthesis stage (V2−V1), while retrieval-sensitive tasks need the persona pushed
   *upstream into the search* — personalized fan-out (V3) adds real value beyond synthesis-only (V3−V2 > 0).
3. **The judge doesn't need the answer key:** re-introducing `latent_profile` (Arm B) barely changes the
   numbers, so the frozen rubric is sufficient — the leak-free judge is the one to report.

*Validity notes:* small n (directional, not powered); `q_8`-style qualitative cases illustrate the
mechanism but are not evidence on their own; `q_3`'s 5 runs are stale (retext) and should be regenerated
before a final table.